## Learning Objective

1) High-Performance LLM Deployment: Deploying quantized Large Language Models (LLMs) using vLLM to optimize inference throughput and GPU memory utilization.
2) Multi-LoRA Architecture: Fine-tuning domain-specific adapters (e.g., SQL, Creative Writing) and implementing a Multi-LoRA serving strategy to dynamically switch between tasks on a single shared base model.

## Installation

Before running the cells below, please complete the following setup steps to ensure the environment is configured correctly.

### 1. Configure the Runtime (Critical)
You must switch to a GPU runtime for this notebook to work.
* Go to the top menu and select **Runtime** $\rightarrow$ **Change runtime type**.
* Under **Hardware accelerator**, select **T4 GPU**.
* Click **Save**.

### 2. Set API Keys
You need to store your Hugging Face token securely.
* Click on the **Keys** icon in the left sidebar of Google Colab.
* Create a new secret named `HF_TOKEN`.
* Paste your Hugging Face Access Token as the value.
* Toggle the **Notebook access** switch to **"On"** so the notebook can read it.

### 3. Weights & Biases (Optional)
This notebook uses **Unsloth**, which integrates with **Weights & Biases (WandB)** for experiment tracking.
* During the training step, you may be prompted to log in to WandB.
* If you have an account, you can link it when prompted.
* If not, you can choose to skip this step or create a free account to track your loss curves.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes xformers
!pip install openai
!pip install fastapi nest-asyncio pyngrok uvicorn
!pip install vllm

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wr24dmhv/unsloth_e877f53f081c44e6a298fda800110191
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wr24dmhv/unsloth_e877f53f081c44e6a298fda800110191
  Resolved https://github.com/unslothai/unsloth.git to commit 5b64813ffb83d6e507989410a64d3b64fbffb323
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 37.3 MB/s eta 0:00:0

In [1]:
!pip install --upgrade --no-deps --force-reinstall unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
  Using cached unsloth_zoo-2026.1.4-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 14.1 MB/s eta 0:00:00
Using cached unsloth_zoo-2026.1.4-py3-none-any.whl (310 kB)
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.1.4
    Uninstalling unsloth_zoo-2026.1.4:
      Successfully uninstalled unsloth_zoo-2026.1.4
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.1.4
    Uninstalling unsloth-2026.1.4:
      Successfully uninstalled unsloth-2026.1.4


## vLLM Server

vLLM is a high-performance, open-source engine designed to serve Large Language Models (LLMs) like Llama 3, Mixtral, and Qwen with extremely high throughput and low latency.

It turns a heavy AI model file (weights) into a fast, web-accessible API server that many users can talk to simultaneously.

### Why is it so fast? (PagedAttention)
Traditional servers waste 60-80% of GPU memory due to fragmentation. vLLM uses **PagedAttention**, an algorithm inspired by operating system virtual memory.
* It breaks memory into non-contiguous "pages."
* It allows the server to batch many more users simultaneously.
* **Result:** 14x–24x higher throughput than standard HuggingFace Transformers.

In [2]:
import subprocess

vllm_process = subprocess.Popen( # run as a background process
        [
            "vllm", "serve",
            "unsloth/llama-3-8b-Instruct-bnb-4bit",
            "--quantization", "bitsandbytes",
            "--load-format", "bitsandbytes",
            "--trust-remote-code",
            "--dtype", "half",

            "--max-model-len", "2048",    # Must fit within memory constraints, safety buffer
            "--gpu-memory-utilization", "0.8", # Reduce base on memory utilization
            "--port", "8000",
            "--enforce-eager",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        start_new_session=True
    )


print(f"vLLM Multi-LoRA Server started with PID: {vllm_process.pid}")

vLLM Multi-LoRA Server started with PID: 2876


In [3]:
import requests
import time
from typing import Tuple
import sys

def check_vllm_status(url: str = "http://localhost:8000/health") -> bool:
    """Check if VLLM server is running and healthy."""
    try:
        response = requests.get(url)
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        return False

def monitor_vllm_process(vllm_process: subprocess.Popen, check_interval: int = 5) -> Tuple[bool, str, str]:
    """
    Monitor VLLM process and return status, stdout, and stderr.
    Returns: (success, stdout, stderr)
    """
    print("Starting VLLM server monitoring...")

    while vllm_process.poll() is None:  # While process is still running
        if check_vllm_status():
            print("✓ VLLM server is up and running!")
            return True, "", ""

        print("Waiting for VLLM server to start...")
        time.sleep(check_interval)

        # Check if there's any output to display
        if vllm_process.stdout.readable(): # standard output
            stdout = vllm_process.stdout.read1().decode('utf-8')
            if stdout:
                print("STDOUT:", stdout)

        if vllm_process.stderr.readable(): # standard error
            stderr = vllm_process.stderr.read1().decode('utf-8')
            if stderr:
                print("STDERR:", stderr)

    # If we get here, the process has ended
    stdout, stderr = vllm_process.communicate()
    return False, stdout.decode('utf-8'), stderr.decode('utf-8')

In [4]:
# Takes around 8 minutes
try:
    success, stdout, stderr = monitor_vllm_process(vllm_process)

    if not success:
        print("\n❌ VLLM server failed to start!")
        print("\nFull STDOUT:", stdout)
        print("\nFull STDERR:", stderr)
        sys.exit(1)

except KeyboardInterrupt:
    print("\n⚠️ Monitoring interrupted by user")
    # # This should just exited the process of probing, not the vllm, if you want it then you could uncomment this.
    # vllm_process.terminate()
    # try:
    #     vllm_process.wait(timeout=5)
    # except subprocess.TimeoutExpired:
    #     vllm_process.kill()

    stdout, stderr = vllm_process.communicate()
    if stdout: print("\nFinal STDOUT:", stdout.decode('utf-8'))
    if stderr: print("\nFinal STDERR:", stderr.decode('utf-8'))
    sys.exit(0)

Starting VLLM server monitoring...
Waiting for VLLM server to start...
STDOUT: (APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325] 
(APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325]        █     █     █▄   ▄█
(APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.15.0
(APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325]   █▄█▀ █     █     █     █  model   unsloth/llama-3-8b-Instruct-bnb-4bit
(APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=2876) INFO 02-04 22:51:12 [utils.py:325] 

STDERR: 2026-02-04 22:50:32.128288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770245432.430128    2876 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770245

In [5]:
import time
import requests
from openai import OpenAI

# --- Wait For Server Initializaion ---
def wait_for_server(url="http://localhost:8000/v1/models"):
    print("Waiting for vLLM to load")
    start = time.time()
    while True:
        try:
            response = requests.get(url)
            if response.status_code == 200:
                print(f"Server is ready! (Took {time.time() - start:.1f}s)")
                return
        except requests.exceptions.ConnectionError:
            pass

        # Check if process died
        if vllm_process.poll() is not None:
            print("Server process died unexpectedly!")
            # Read error logs from the pipe if you used PIPE (warning: this can hang if buffer is full)
            out, err = vllm_process.communicate()
            print(err.decode())
            break

        time.sleep(5)

# Wait before connecting
wait_for_server()

Waiting for vLLM to load
Server is ready! (Took 0.0s)


In [7]:
# --- Initialize Client ---
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")


try:
    response = client.completions.create(
        model="unsloth/llama-3-8b-Instruct-bnb-4bit",
        prompt="what is the colour of the sky",
        max_tokens=100,
        temperature=0.1,
        stop=["###"]             # Stop it from hallucinating new instructions
    )
    print(f"Output:\n{response.choices[0].text.strip()}")
except Exception as e:
    print(f"Error: {e}")

SQL Output:
?
The colour of the sky is a fascinating topic that has been debated by scientists and philosophers for centuries. The answer, however, is quite simple: the colour of the sky is blue.

But why is the sky blue? Well, it's because of a phenomenon called Rayleigh scattering, which is the scattering of light by small particles or molecules in the atmosphere. When sunlight enters the Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).


In [8]:
# To terminate (stop) the vLLM server process that is running in the background
!pkill vllm

## Low-Rank Adaptation (LoRA)

**LoRA** is a popular parameter-efficient fine-tuning (PEFT) technique used to adapt large pre-trained models (like GPT-4, Llama 3) to specific tasks or styles without the massive computational cost of retraining the entire model.

### 1. The Problem: Full Fine-Tuning
Traditional fine-tuning requires updating **every single parameter** (weight) in the neural network.

* **Memory Constraints:** For a model with billions of parameters, this requires massive amounts of GPU memory (VRAM) to store optimizer states and gradients (often 3-4x the model size).
* **Storage Issues:** After training, the new model is just as large as the original (e.g., 100GB), making it difficult to store or share multiple versions.

**Mathematical Context:**
If $W_{pre}$ is the pre-trained weight and $W_{ft}$ is the fine-tuned weight, we are trying to learn the full difference matrix $\Delta W$:
$$W_{pre} → W_{ft} $$
$$W_{ft} - W_{pre} = \Delta W$$

### 2. The Solution: Low-Rank Decomposition
LoRA proposes that the change in weights ($\Delta W$) is "low-rank" which means it doesn't need high complexity (full dimension) to be effective.

Instead of updating the massive original matrix $W$, LoRA freezes it and adds a parallel branch of two tiny matrices, **$A$** and **$B$**.

**How it works:**
1.  **Freeze** the pre-trained weight matrix $W$ (it remains untouched).
2.  **Inject** two trainable low-rank matrices, $A$ and $B$, into the layer.
$$
B \in \mathbb{R}^{n_{out} \times r}, \quad A \in \mathbb{R}^{r \times n_{in}}
$$
$$
r \ll n_{in}, \quad r \ll n_{out}
$$

* **$r$ (Rank):** A hyperparameter (e.g., 8, 16, 64) that determines the size of $A$ and $B$. A lower rank means fewer parameters to train.

3.  **Compute** the output ($h$) by adding the original path and the LoRA path:

$$h = W_0 x + \Delta W x$$
$$h = W_0 x + B A x$$

* **Result:** You only train $A$ and $B$. The original $W$ remains untouched.

### 3. Key Benefits
* **Hardware Efficiency:** Reduces the number of trainable parameters by up to 10,000x. You can fine-tune huge models on consumer hardware (e.g., a single NVIDIA RTX 3090 or 4090).
* **Lower Memory Cost:** Significantly decreases VRAM usage during training (e.g., avoids storing optimizer states for the full model).
* **Compact Storage:** LoRA weights are tiny. A LoRA file might be **10MB – 200MB**, whereas a full model checkpoint is **20GB – 100GB**.
* **Modularity:** You can keep one "Base Model" frozen and swap different LoRAs in and out on the fly. (e.g., Switch from a "Coding LoRA" to a "Poetry LoRA" instantly, without needing to host 2 full models).
* **Zero Inference Latency:** If you merge the LoRA weights back into the base model ($W_{new} = W + BA$), the inference speed is identical to the original model.

## Training LoRA Adaptor

This script demonstrates how to efficiently fine-tune the **Llama 3 8B** model using the `unsloth` library and Low-Rank Adaptation (LoRA). It is optimized for free-tier GPUs (like Google Colab T4).

#### Key Steps performed in the code:
1.  **Load Model:** Initializes a 4-bit quantized version of `llama-3-8b` using `FastLanguageModel` for memory efficiency.
- [Link to Model: unsloth/llama-3-8b-bnb-4bit](https://huggingface.co/unsloth/llama-3-8b-bnb-4bit)
2.  **Attach LoRA:** Injects LoRA adapters into the model's projection layers (query, key, value, output, etc.) with a rank ($r$) of 16. This ensures we only train a tiny fraction of the parameters.
3.  **Prepare Data:** Loads a sample SQL dataset (`b-mc2/sql-create-context`) and formats it using the standard **Alpaca** prompt template (Instruction/Input/Response).
- [Link to Dataset: b-mc2/sql-create-context](https://huggingface.co/datasets/b-mc2/sql-create-context)
4.  **Train:** Sets up the `SFTTrainer` (Supervised Fine-Tuning) with optimized hyperparameters (AdamW 8-bit optimizer, gradient accumulation) and runs a short training session.
5.  **Save:** Saves the trained LoRA adapters and tokenizer to the local `./sql_adapter` directory for future use.


### Hugging Face Gated Model
- Gated Models:
  - Example [meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) where Meta releases Llama models under a custom "Llama 3.1 Community License."
  - Acceptance of the License Agreement, key terms includes:
    - Commercial Use: The model is free for most users, but if you have more than 700 million monthly active users, you must request a separate license from Meta.
    - Attribution: Any derivative works or products built using the model must include "Built with Llama" or start the model name with "Llama."
    - Liability: You agree that Meta is not responsible for the outputs the model generates or how you use them.
  - Acceptable Use Policy (AUP): not use for illegal activites
  - Request for access usually automated and can take anywhere from a few min/hour
- Non-gated Models:
  - Download immediately but license still applies (check model details and intended use)

In [9]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# --- Configuration ---
max_seq_length = 2048 # Set the maximum number of "tokens" the model can read and generate in a single go
dtype = None # Determine numerical precisio of weights,auto detection based on GPU capabilities
load_in_4bit = True # Enable 4-bit quantization, instead of storing weights as 16-bit

# --- Load Model ---
print("Loading Unsloth Llama 3 Model")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- Attach LoRA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank mentioned above
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Attention and FFN layers
    lora_alpha = 16, # scaling factor for weight update
    lora_dropout = 0, # 0.05 or 0.1
    bias = "none", # not training bias param, saving on parameters
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.ddpo_trainer: Failed to import trl.trainer.ddpo_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.ip_adapter because of the following error (look up to see its traceback):
/usr/local/lib/python3.12/dist-packages/xformers/flash_attn_3/_C.so: undefined symbol: torch_list_push_back
Loading Unsloth Llama 3 Model
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.15.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [10]:
# --- Prepare Data ---
# https://github.com/tatsu-lab/stanford_alpaca
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

dataset = load_dataset("b-mc2/sql-create-context", split = "train[:100]") # Small subset for demo
def formatting_prompts_func(examples):
    instructions = examples["question"]
    inputs       = examples["context"]
    outputs      = examples["answer"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [11]:
print(dataset["text"][0])

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
How many heads of the departments are older than 56 ?

### Input:
CREATE TABLE head (age INTEGER)

### Response:
SELECT COUNT(*) FROM head WHERE age > 56<|end_of_text|>


### Training Terminology & Math

* **Epoch:** One complete pass through your entire training dataset.
* **Step:** One single update of the model’s weights.
* **Effective Batch Size:**
    `per_device_train_batch_size` * `gradient_accumulation_step`
    = 2 * 4 = **8** (1 step = 8 samples)
* **Max Steps:**
    30 steps * 8 samples/step = **240 samples**
    *(Note: Because our training set has 100 rows, 30 steps == 240/100 = 2.4 epochs)*

In [12]:
# --- Train ---
print("Starting Training")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Dataset number of processes, how many CPU cores used in parallel to process
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Feed 2 example to GPU at a time, keep VRAM low
        gradient_accumulation_steps = 4, # accumulate result of 4 batches before update model weights
        max_steps = 30, # Very short for demo
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), # hardware compatability
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # 8-bit quantization for optimizer states, saving ~75% memory
        output_dir = "outputs",
    ),
)
trainer.train() # Approx 2mins

# --- Save ---
print("Saving LoRA Adapter to ./sql_adapter")
model.save_pretrained("sql_adapter")
tokenizer.save_pretrained("sql_adapter")
print("Done!")

Starting Training


Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

...

wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alice-chua (alice-chua-university-of-toronto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.367100
2,2.240700
3,2.518100
4,2.073800
5,2.075600
6,2.119900
7,1.908800
8,2.025600
9,1.916700
10,1.877000


Saving LoRA Adapter to ./sql_adapter
Done!


## Clearing GPU Memory

Training session leave behind "computational leftovers" like the optimizer state, gradients, activations, which persists in the VRAM even after the code stop running.

In [13]:
import gc
import torch

# Delete variables
try:
    del model
    del tokenizer
    del trainer
except:
    pass

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

# Double check memory
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Memory cleared. Ready for vLLM.")

GPU Memory Allocated: 1.12 GB
Memory cleared. Ready for vLLM.


In [14]:
def trace_gpu_memory():
    # Force garbage collection first to clear dead variables
    gc.collect()
    torch.cuda.empty_cache()

    # List to store (size_in_mb, shape, object_type)
    tensors = []

    # Iterate over all objects in memory
    for obj in gc.get_objects():
        try:
            # Check if it's a Tensor or a Parameter on CUDA
            if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                if obj.is_cuda:
                    size_mb = (obj.element_size() * obj.nelement()) / (1024 ** 2)
                    tensors.append((size_mb, list(obj.size()), type(obj)))
        except:
            pass # Ignore objects that fail access

    # Sort by size (largest first)
    tensors.sort(key=lambda x: x[0], reverse=True)

    # Print Top 10
    print(f"{'SIZE (MB)':<12} | {'SHAPE':<25} | {'TYPE'}")
    print("-" * 60)
    total_tracked = 0
    for size, shape, type_ in tensors[:10]:
        total_tracked += size
        print(f"{size:<12.2f} | {str(shape):<25} | {type_.__name__}")

    print("-" * 60)
    print(f"Total explicitly tracked tensors: {total_tracked:.2f} MB")
    print(f"Total PyTorch Allocated:          {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

trace_gpu_memory()

/tmp/ipython-input-3697235734.py:13: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):


WARNING 02-04 23:08:54 [interface.py:598] Current platform cuda does not have 'data' attribute.
WARNING 02-04 23:08:54 [interface.py:598] Current platform cuda does not have 'data' attribute.


/usr/local/lib/python3.12/dist-packages/tensorboard/data/server_ingester.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


SIZE (MB)    | SHAPE                     | TYPE
------------------------------------------------------------
2004.00      | [4096, 128256]            | FakeTensor
2004.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [4096, 128256]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | Parameter
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [4096, 128256]            | FakeTensor
1002.00      | [128256, 4096]            | FakeTensor
1002.00      | [128256, 4096]            | Tensor
------------------------------------------------------------
Total explicitly tracked tensors: 12024.00 MB
Total PyTorch Allocated:          1142.91 MB


## Training a second LoRA Adaptor

This section demonstrates how to train a **second, distinct adapter** on the same base model. While the first adapter was for SQL generation, this one fine-tunes the model to mimic **Shakespearean writing**.

### Key Steps:
1.  **Reload Base Model:** We re-initialize the Llama 3 model. This is necessary to attach a fresh, blank set of LoRA adapters (starting from scratch rather than continuing from the SQL training).
2.  **Load Creative Dataset:** We load the `Trelis/tiny-shakespeare` dataset, which contains lines from Shakespeare's plays.
    * [Link to Dataset: Trelis/tiny-shakespeare](https://huggingface.co/datasets/Trelis/tiny-shakespeare)
3.  **Format Data:** We apply a custom formatting function to prompt the model with a specific style trigger: `### Style: Shakespeare`.
4.  **Train:** We run the `SFTTrainer` again with memory-optimized settings (Batch size 2, Gradient Accumulation 4) for 30 steps.
5.  **Save to New Directory:** The trained weights are saved to a separate folder (`./creative_adapter`), ensuring they don't overwrite the previous SQL adapter.
6.  **Final Cleanup:** The script explicitly deletes the model and clears GPU cache (`gc.collect()`, `torch.cuda.empty_cache()`) to free up resources for inference or further tasks.

In [15]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# --- Reload Base Model ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    random_state = 3407,
)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.15.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [16]:
# --- Load Creative Dataset ---
dataset = load_dataset("Trelis/tiny-shakespeare", split="train[:100]")

# Simple formatting for raw text to instruction style
def formatting_func(examples):
    text = examples["Text"]
    # if "Text" not in examples and "text" in examples:
    #      text = examples["text"]
    return { "text" : [f"### Style: Shakespeare\n### Content: {t}" for t in text] }

dataset = dataset.map(formatting_func, batched = True)

README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
print(dataset["text"][0])

### Style: Shakespeare
### Content: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for

In [18]:
# --- Train ---
print("Starting Training")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_creative",
    ),
)
trainer.train() # Approx 4 mins

# --- Save to New Directory ---
print("Saving Creative Adapter to ./creative_adapter...")
model.save_pretrained("creative_adapter")
tokenizer.save_pretrained("creative_adapter")

Starting Training


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.347800
2,1.319500
3,1.317000
4,1.346400
5,1.340100
6,1.248200
7,1.260400
8,1.197400
9,1.295200
10,1.311000


Saving Creative Adapter to ./creative_adapter...


('creative_adapter/tokenizer_config.json',
 'creative_adapter/special_tokens_map.json',
 'creative_adapter/tokenizer.json')

In [19]:
# --- Final Cleanup ---
del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()

## vLLM Server using LoRA

### Multi-LoRA Support
In this notebook, we are using vLLM's **Multi-LoRA** feature.
* **One Base Model:** We load the heavy Llama-3-8B model once.
* **Multiple Adapters:** We can serve both the `sql_adapter` and `creative_adapter` on the same server.
* **Dynamic Switching:** The server swaps the tiny adapter weights on-the-fly depending on which `model` name you request in the API.

### OpenAI Compatible API
vLLM provides an endpoint that looks exactly like OpenAI's. You can use standard libraries to interact with it:

```python
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# Request the SQL Adapter
response = client.chat.completions.create(
    model="sql-model",
    messages=[...]
)

In [ ]:
# Find and kill any process using the GPU
!fuser -k -v /dev/nvidia0

In [ ]:
# Check if any processes are running, because we need min 12GB RAM to run the server
!nvidia-smi

In [2]:
import subprocess

vllm_process = subprocess.Popen( # run as a background process
        [
            "vllm", "serve",
            "unsloth/llama-3-8b-bnb-4bit",  # The shared Base Model
            "--quantization", "bitsandbytes",
            "--load-format", "bitsandbytes",
            "--trust-remote-code",
            "--dtype", "half",
            "--enable-lora", # Initialize LoRA management system

            # Syntax: name_in_api=path_to_folder
            "--lora-modules",
            "sql-model=./sql_adapter",
            "shakespeare-model=./creative_adapter",

            "--max-loras", "2",           # Allow switching between 2 adapters
            "--max-model-len", "2048",    # Must fit within memory constraints, safety buffer
            "--gpu-memory-utilization", "0.8", # Reduce base on memory utilization
            "--port", "8000",
            "--enforce-eager",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        start_new_session=True
    )


print(f"vLLM Multi-LoRA Server started with PID: {vllm_process.pid}")

vLLM Multi-LoRA Server started with PID: 10101


In [3]:
import requests
import time
from typing import Tuple
import sys

def check_vllm_status(url: str = "http://localhost:8000/health") -> bool:
    """Check if VLLM server is running and healthy."""
    try:
        response = requests.get(url)
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        return False

def monitor_vllm_process(vllm_process: subprocess.Popen, check_interval: int = 5) -> Tuple[bool, str, str]:
    """
    Monitor VLLM process and return status, stdout, and stderr.
    Returns: (success, stdout, stderr)
    """
    print("Starting VLLM server monitoring...")

    while vllm_process.poll() is None:  # While process is still running
        if check_vllm_status():
            print("✓ VLLM server is up and running!")
            return True, "", ""

        print("Waiting for VLLM server to start...")
        time.sleep(check_interval)

        # Check if there's any output to display
        if vllm_process.stdout.readable(): # standard output
            stdout = vllm_process.stdout.read1().decode('utf-8')
            if stdout:
                print("STDOUT:", stdout)

        if vllm_process.stderr.readable(): # standard error
            stderr = vllm_process.stderr.read1().decode('utf-8')
            if stderr:
                print("STDERR:", stderr)

    # If we get here, the process has ended
    stdout, stderr = vllm_process.communicate()
    return False, stdout.decode('utf-8'), stderr.decode('utf-8')

In [4]:
try:
    success, stdout, stderr = monitor_vllm_process(vllm_process)

    if not success:
        print("\n❌ VLLM server failed to start!")
        print("\nFull STDOUT:", stdout)
        print("\nFull STDERR:", stderr)
        sys.exit(1)

except KeyboardInterrupt:
    print("\n⚠️ Monitoring interrupted by user")
    # # This should just exited the process of probing, not the vllm, if you want it then you could uncomment this.
    # vllm_process.terminate()
    # try:
    #     vllm_process.wait(timeout=5)
    # except subprocess.TimeoutExpired:
    #     vllm_process.kill()

    stdout, stderr = vllm_process.communicate()
    if stdout: print("\nFinal STDOUT:", stdout.decode('utf-8'))
    if stderr: print("\nFinal STDERR:", stderr.decode('utf-8'))
    sys.exit(0)

Starting VLLM server monitoring...
Waiting for VLLM server to start...
STDOUT: (APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325] 
(APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325]        █     █     █▄   ▄█
(APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.15.0
(APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325]   █▄█▀ █     █     █     █  model   unsloth/llama-3-8b-bnb-4bit
(APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=10101) INFO 02-04 23:16:42 [utils.py:325] 

STDERR: 2026-02-04 23:16:01.009926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770246961.065571   10101 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770246961

## Querying the server

In [5]:
import time
import requests
from openai import OpenAI

# --- Wait For Server Initializaion ---
def wait_for_server(url="http://localhost:8000/v1/models"):
    print("Waiting for vLLM to load")
    start = time.time()
    while True:
        try:
            response = requests.get(url)
            if response.status_code == 200:
                print(f"Server is ready! (Took {time.time() - start:.1f}s)")
                return
        except requests.exceptions.ConnectionError:
            pass

        # Check if process died
        if vllm_process.poll() is not None:
            print("Server process died unexpectedly!")
            # Read error logs from the pipe if you used PIPE (warning: this can hang if buffer is full)
            out, err = vllm_process.communicate()
            print(err.decode())
            break

        time.sleep(5)

# Wait before connecting
wait_for_server()

# --- Initialize Client ---
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# --- Test SQL LoRA ---
print("\n--- Testing SQL LoRA ---")

prompt_sql = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Get all users from Canada.

### Input:


### Response:
"""

try:
    response = client.completions.create(
        model="sql-model",       # Call the LoRA
        prompt=prompt_sql,       # Send the raw training format
        max_tokens=100,
        temperature=0.1,
        stop=["###"]             # Stop it from hallucinating new instructions
    )
    print(f"SQL Output:\n{response.choices[0].text.strip()}")
except Exception as e:
    print(f"Error: {e}")

# --- Test Shakespeare LoRA ---
print("\n--- Testing Shakespeare LoRA ---")
prompt_shake = "### Style: Shakespeare\n### Content: Explain the concept of love."

try:
    response = client.completions.create(
        model="shakespeare-model",
        prompt=prompt_shake,
        max_tokens=100,
        temperature=0.7
    )
    print(f"Shakespeare Output:\n{response.choices[0].text.strip()}")
except Exception as e:
    print(f"Error: {e}")

Waiting for vLLM to load
Server is ready! (Took 0.0s)

--- Testing SQL LoRA ---
SQL Output:
SELECT * FROM users WHERE country = 'Canada';

--- Testing Shakespeare LoRA ---
Shakespeare Output:
Provide examples of love in Shakespeare's plays and explain what each example means.

## Love in Shakespeare's plays

Love in Shakespeare's plays is often a complex and nuanced concept. It can be seen as a powerful force that can bring people together or tear them apart, and it can be a source of both happiness and pain. In Shakespeare's works, love is often depicted as a force that can be both positive and negative, depending on the circumstances.

One example of love in Shakespeare's plays is in


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from openai import OpenAI
import time

# --- Initialize Client ---
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# --- Formatter ---
def format_prompt(model_key, user_input):
    if model_key == "sql-model":
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{user_input}

### Input:


### Response:
"""
    elif model_key == "shakespeare-model":
        return f"### Style: Shakespeare\n### Content: {user_input}"
    else:
        return user_input

# --- Create Interactive Widget to test vLLM Server---
style = {'description_width': 'initial'}

header = widgets.HTML("<h2>🧪 LoRA Adapter Playground</h2>")

model_dropdown = widgets.Dropdown(
    options=[
        ('📊 SQL Generator', 'sql-model'),
        ('🎭 Shakespeare', 'shakespeare-model'),
        ('🧠 Base Model (Raw)', 'unsloth/llama-3-8b-bnb-4bit')
    ],
    value='sql-model',
    description='<b>Select Adapter:</b>',
    style=style,
    layout=widgets.Layout(width='50%')
)

input_box = widgets.Textarea(
    value='',
    placeholder='Type your instruction here...',
    description='<b>Your Prompt:</b>',
    style=style,
    layout=widgets.Layout(width='80%', height='100px')
)

# Button with initial state
send_btn = widgets.Button(
    description="Generate Response",
    button_style='primary',
    icon='bolt',
    layout=widgets.Layout(width='20%')
)

out = widgets.Output(layout={'border': '1px solid #444', 'padding': '15px', 'margin_top': '10px'})

def on_click(b):
    prompt_text = input_box.value
    if not prompt_text:
        with out:
            clear_output()
            print("⚠️ Please enter a prompt first.")
        return

    # --- UI STATE: BUSY ---
    # Disable button and change text to show we are working
    send_btn.disabled = True
    send_btn.description = "Running..."
    send_btn.button_style = 'warning' # Turns Orange

    with out:
        clear_output()
        # Show a loading message
        print(f"⏳ Sending query to {model_dropdown.label}...")
        print("   (This might take 5-10 seconds depending on GPU load)")

    try:
        # Prepare Prompt
        full_prompt = format_prompt(model_dropdown.value, prompt_text)

        # API Call
        start_time = time.time()
        response = client.completions.create(
            model=model_dropdown.value,
            prompt=full_prompt,
            max_tokens=200,
            temperature=0.1 if "sql" in model_dropdown.value else 0.7,
            stop=["###"]
        )
        duration = time.time() - start_time

        # --- UI STATE: SUCCESS ---
        with out:
            clear_output()
            print(f"✅ Complete ({duration:.2f}s)\n")
            print("✨ RESPONSE:")
            print("-" * 60)
            print(response.choices[0].text.strip())
            print("-" * 60)

    except Exception as e:
        # --- UI STATE: ERROR ---
        with out:
            print(f"\n❌ Error: {e}")

    finally:
        # --- UI STATE: RESET ---
        # Always re-enable the button, even if it crashed
        send_btn.disabled = False
        send_btn.description = "Generate Response"
        send_btn.button_style = 'primary' # Back to Blue

send_btn.on_click(on_click)

display(header, model_dropdown, input_box, send_btn, out)

HTML(value='<h2>🧪 LoRA Adapter Playground</h2>')

Dropdown(description='<b>Select Adapter:</b>', layout=Layout(width='50%'), options=(('📊 SQL Generator', 'sql-m…

Textarea(value='', description='<b>Your Prompt:</b>', layout=Layout(height='100px', width='80%'), placeholder=…

Button(button_style='primary', description='Generate Response', icon='bolt', layout=Layout(width='20%'), style…

Output(layout=Layout(border='1px solid #444', padding='15px'))

Additional task: To modify the notebook to use [unsloth/llama-3-8b-Instruct-bnb-4bit](https://huggingface.co/unsloth/llama-3-8b-Instruct-bnb-4bit) instead

## Final Cleanup

In [ ]:
# To terminate (stop) the vLLM server process that is running in the background
!pkill vllm